
# CZSO IKZ – workflow od A do Z (v5: přesnější roky + filtrované previewe)

Notebook dělá celý pipeline v jednom místě:

1. načte `Katalog_trideny`,
2. vybere všechny relevantní datasety (vše kromě `referenční / technická`),
3. stáhne / profiluje každý dataset,
4. zařadí ho do kategorií **A–D**,
5. uloží `head()` + `tail()` previewe a manifest,
6. umožní výstupy zobrazit interaktivně po kategoriích.

## Co je v této verzi opravené

- **přesnější detekce roků** – už se nechytají falešné roky ze sloupce `hodnota`
- **priorita skutečného časového sloupce** – `rok`, `year`, `datum`, `casref`, `obdobi` mají přednost
- **heuristika unikátnosti / podílu zásahů** – sloupec s časem musí mít omezený počet unikátních roků a silný časový signál
- **previewe jsou filtrované ještě před `head()` a `tail()`** – pokud je nalezen řádkový časový sloupec, `head` i `tail` se tvoří už jen z `target years`
- **stažené zdrojové soubory se po preview ve výchozím stavu mažou**

> Výchozí cílové roky jsou **2023 a 2024**.
> Výchozí logika je **přísná**: dataset musí obsahovat **oba roky**.
> Výchozí chování je, že `HEAD` a `TAIL` jsou filtrovány jen na `target years`.


In [1]:

from pathlib import Path
import pandas as pd

from czso_ikz_a_to_z_v5 import (
    read_catalogue,
    select_relevant_datasets,
    process_relevant_catalogue,
    build_summary_tables,
    display_category_results,
)

BASE_DIR = Path.cwd()
CATALOG_XLSX = BASE_DIR / 'czso_katalog_trideny.xlsx'
OUTPUT_DIR = BASE_DIR / 'czso_ikz_a_to_z_output'

TARGET_YEARS = [2023, 2024]
REQUIRE_ALL_TARGET_YEARS = True   # False = stačí libovolný překryv s oknem 2023–2024
FILTER_PREVIEW_TO_TARGET_YEARS = True
HEAD_ROWS = 5
TAIL_ROWS = 5
MAX_DATASETS = None               # např. 20 pro rychlý test
FORCE_REDOWNLOAD = False
DELETE_DOWNLOADED_FILES_AFTER_PREVIEW = True
PAUSE_SECONDS = 0.0

DISPLAY_CATEGORIES = None         # např. ['A'] nebo ['A', 'B']
DISPLAY_LIMIT = None              # např. 10 pro rychlý náhled

print('Base dir :', BASE_DIR)
print('Catalog  :', CATALOG_XLSX)
print('Output   :', OUTPUT_DIR)
print('Years    :', TARGET_YEARS)
print('Strict   :', REQUIRE_ALL_TARGET_YEARS)
print('Filter preview rows to target years :', FILTER_PREVIEW_TO_TARGET_YEARS)


Base dir : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_a_to_z_bundle_v5_year_filter
Catalog  : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_a_to_z_bundle_v5_year_filter\czso_katalog_trideny.xlsx
Output   : C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_a_to_z_bundle_v5_year_filter\czso_ikz_a_to_z_output
Years    : [2023, 2024]
Strict   : True
Filter preview rows to target years : True


In [2]:

catalogue_df = read_catalogue(CATALOG_XLSX)
relevant_df = select_relevant_datasets(catalogue_df)

print('Celkem řádků v katalogu:', len(catalogue_df))
print('Relevantních datasetů po odfiltrování technických:', len(relevant_df))

summary_relevance = (
    relevant_df.groupby('relevance_pro_IKZ', dropna=False)
    .size()
    .reset_index(name='pocet_datasetu')
    .sort_values('pocet_datasetu', ascending=False)
    .reset_index(drop=True)
)
summary_relevance


Celkem řádků v katalogu: 1030
Relevantních datasetů po odfiltrování technických: 656


,relevance_pro_IKZ,pocet_datasetu
0,volitelná / specializovaná,390
1,jádrová pro IKŽ,197
2,podpůrná / kontextová,69


In [3]:

relevant_df[['dataset_id', 'dataset_title', 'relevance_pro_IKZ', 'hlavni_skupina', 'detailni_skupina']].head(20)


,dataset_id,dataset_title,relevance_pro_IKZ,hlavni_skupina,detailni_skupina
0,200068,Dokončené byty v obcích,jádrová pro IKŽ,Bydlení a technická infrastruktura,bytová výstavba a stavebnictví
1,sldb2021_byty_obydlenost,Sčítání 2021 - Byty podle obydlenosti,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
2,sldb2021_byty_obydlenost_druhdomu,Sčítání 2021 - Byty podle obydlenosti a druhu ...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
3,sldb2021_byty_plocha,Sčítání 2021 - Obydlené byty podle celkové plo...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
4,sldb2021_byty_energie,Sčítání 2021 - Obydlené byty podle hlavního zd...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
5,sldb2021_obybyty_material_druhdomu,Sčítání 2021 - Obydlené byty podle materiálu n...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
6,sldb2021_obybyty_obdobi_druhdomu,Sčítání 2021 - Obydlené byty podle období výst...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
7,sldb2021_obybyty_poloha_druhdomu,Sčítání 2021 - Obydlené byty podle polohy bytu...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
8,sldb2021_byty_mistnosti,Sčítání 2021 - Obydlené byty podle počtu obytn...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení
9,sldb2021_byty_mistnosti_kuchyne,Sčítání 2021 - Obydlené byty podle počtu obytn...,jádrová pro IKŽ,Bydlení a technická infrastruktura,byty a kvalita bydlení


In [4]:

manifest_df, results = process_relevant_catalogue(
    CATALOG_XLSX,
    output_dir=OUTPUT_DIR,
    target_years=TARGET_YEARS,
    require_all_target_years=REQUIRE_ALL_TARGET_YEARS,
    head_rows=HEAD_ROWS,
    tail_rows=TAIL_ROWS,
    max_datasets=MAX_DATASETS,
    force_redownload=FORCE_REDOWNLOAD,
    cleanup_downloaded_files=DELETE_DOWNLOADED_FILES_AFTER_PREVIEW,
    pause_seconds=PAUSE_SECONDS,
    display_in_notebook=False,
    filter_preview_to_target_years=FILTER_PREVIEW_TO_TARGET_YEARS,
)

print('Počet zpracovaných datasetů:', len(manifest_df))
manifest_df.head()


Dataset: 200068 :: Dokončené byty v obcích
status   : OK_LKOD
category : A (obsahuje cílové roky (data) a má primary key)
pk       : True | vuzemi_cis | equivalent
years    : [2023, 2024] missing []
error    : roky načteny z celého souboru po chunkách | head/tail filtrovány na target years [2023, 2024] přes sloupec `rok`; matching_rows=28635 | Archiv obsahuje 1 čitelných souborů; použit byl vybraný soubor `OD_STA01_2025090509313708.CSV`.
Dataset: sldb2021_byty_obydlenost :: Sčítání 2021 - Byty podle obydlenosti
status   : OK_LKOD
category : D (nesplňuje podmínky A/B/C)
pk       : False | uzemi_cis | territorial_non_municipal
years    : [] missing [2023, 2024]
error    : roky načteny z celého souboru po chunkách | head/tail filtrovány na target years [2023, 2024] přes sloupec `sldb_rok`; matching_rows=0
Dataset: sldb2021_byty_obydlenost_druhdomu :: Sčítání 2021 - Byty podle obydlenosti a druhu domu
status   : OK_LKOD
category : D (nesplňuje podmínky A/B/C)
pk       : False | uzemi_cis |

C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_a_to_z_bundle_v5_year_filter\_czso_ikz_a_to_z_v4_base.py:1333: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  manifest_df["_has_pk_sort"] = manifest_df["has_primary_key"].fillna(False).astype(int) * -1
C:\Users\jakub.sekanina\projekt_index_kvality_cr\Datamining-2026\czso_ikz_a_to_z_bundle_v5_year_filter\_czso_ikz_a_to_z_v4_base.py:1334: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  manifest_df["_has_years_sort"] = manifest_df["has_target_years"].fillna(False).astype(int) * -1


Počet zpracovaných datasetů: 656


,order,category,category_reason,relevance_pro_ikz,relevance_rank,dataset_id,dataset_title,rodina_datasetu,hlavni_skupina,detailni_skupina,...,primary_key_kind,primary_key_note,start_catalog,end_catalog,periodicita_text,head_csv,tail_csv,metadata_json,page,error
0,1,A,obsahuje cílové roky (data) a má primary key,jádrová pro IKŽ,0,200068,Dokončené byty v obcích,Dokončené byty v obcích,Bydlení a technická infrastruktura,bytová výstavba a stavebnictví,...,equivalent,územní kód použitelný jako obecní ekvivalent (...,1997-01-01 00:00:00,2024-12-31 00:00:00,roční,previews\A_200068_Dokoncene_byty_v_obcich_head...,previews\A_200068_Dokoncene_byty_v_obcich_tail...,previews\A_200068_Dokoncene_byty_v_obcich_meta...,https://csu.gov.cz/docs/107508/b8162c4b-9666-e...,roky načteny z celého souboru po chunkách | he...
1,2,A,obsahuje cílové roky (data) a má primary key,jádrová pro IKŽ,0,130149,Obyvatelstvo podle pohlaví a základních věkový...,Obyvatelstvo podle pohlaví a základních věkový...,Demografie a populace,věková a pohlavní struktura populace,...,equivalent,územní kód použitelný jako obecní ekvivalent (...,2000-01-01 00:00:00,2024-12-31 00:00:00,roční,previews\A_130149_Obyvatelstvo_podle_pohlavi_a...,previews\A_130149_Obyvatelstvo_podle_pohlavi_a...,previews\A_130149_Obyvatelstvo_podle_pohlavi_a...,https://csu.gov.cz/docs/107508/3de428e4-62ea-3...,roky načteny z celého souboru po chunkách | he...
2,3,A,obsahuje cílové roky (data) a má primary key,podpůrná / kontextová,1,270228,Osevní plochy zemědělských plodin podle krajů,Osevní plochy zemědělských plodin podle krajů,"Životní prostředí, zemědělství a odpady",zemědělství a lesnictví,...,equivalent,územní kód použitelný jako obecní ekvivalent (...,2014-01-01 00:00:00,2025-12-31 00:00:00,roční,previews\A_270228_Osevni_plochy_zemedelskych_p...,previews\A_270228_Osevni_plochy_zemedelskych_p...,previews\A_270228_Osevni_plochy_zemedelskych_p...,https://csu.gov.cz/docs/107508/dc16980f-c030-9...,roky načteny z celého souboru po chunkách | he...
3,4,A,obsahuje cílové roky (data) a má primary key,podpůrná / kontextová,1,260017,Pracovní neschopnost pro nemoc a úraz podle ok...,Pracovní neschopnost pro nemoc a úraz podle ok...,Zdraví a sociální služby,pracovní neschopnost,...,equivalent,územní kód použitelný jako obecní ekvivalent (...,2006-01-01 00:00:00,2025-06-30 00:00:00,roční,previews\A_260017_Pracovni_neschopnost_pro_nem...,previews\A_260017_Pracovni_neschopnost_pro_nem...,previews\A_260017_Pracovni_neschopnost_pro_nem...,https://csu.gov.cz/docs/107508/fc146a4a-7550-9...,roky načteny z celého souboru po chunkách | he...
4,5,A,obsahuje cílové roky (data) a má primary key,podpůrná / kontextová,1,200077,Stavební povolení,Stavební povolení,Bydlení a technická infrastruktura,bytová výstavba a stavebnictví,...,equivalent,územní kód použitelný jako obecní ekvivalent (...,2005-01-01 00:00:00,2025-12-31 00:00:00,měsíční,previews\A_200077_Stavebni_povoleni_head.csv,previews\A_200077_Stavebni_povoleni_tail.csv,previews\A_200077_Stavebni_povoleni_meta.json,https://csu.gov.cz/docs/107508/130319fd-bc2c-2...,roky načteny z celého souboru po chunkách | he...


In [5]:

summary_tables = build_summary_tables(manifest_df)
summary_tables['datasets_by_category']


,category,relevance_pro_ikz,pocet_datasetu
0,A,podpůrná / kontextová,16
1,A,jádrová pro IKŽ,2
2,B,podpůrná / kontextová,12
3,B,volitelná / specializovaná,9
4,C,jádrová pro IKŽ,46
5,C,podpůrná / kontextová,3
6,D,volitelná / specializovaná,381
7,D,jádrová pro IKŽ,149
8,D,podpůrná / kontextová,38


In [6]:

summary_tables['datasets_by_pk_and_years']


,has_primary_key,has_target_years,pocet_datasetu
0,True,True,18
1,True,False,49
2,False,True,21
3,False,False,269
4,NaN,True,14
5,NaN,False,282
6,NaN,NaN,3


In [7]:

summary_tables['datasets_by_status']


,status,pocet_datasetu
0,OK_LKOD,363
1,UNSUPPORTED_FORMAT,256
2,NO_SELECTION,37


In [8]:

manifest_df[['order', 'category', 'dataset_id', 'dataset_title', 'relevance_pro_ikz', 'has_primary_key', 'primary_key_column', 'has_target_years', 'target_years_present', 'years_sample', 'status']].head(40)


,order,category,dataset_id,dataset_title,relevance_pro_ikz,has_primary_key,primary_key_column,has_target_years,target_years_present,years_sample,status
0,1,A,200068,Dokončené byty v obcích,jádrová pro IKŽ,True,vuzemi_cis,True,"2023, 2024","1997, 1998, 1999, 2000, 2001, 2002 ... 2019, 2...",OK_LKOD
1,2,A,130149,Obyvatelstvo podle pohlaví a základních věkový...,jádrová pro IKŽ,True,uzemi_cis,True,"2023, 2024","2000, 2001, 2002, 2003, 2004, 2005 ... 2019, 2...",OK_LKOD
2,3,A,270228,Osevní plochy zemědělských plodin podle krajů,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021...",OK_LKOD
3,4,A,260017,Pracovní neschopnost pro nemoc a úraz podle ok...,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2006, 2007, 2008, 2009, 2010, 2011 ... 2020, 2...",OK_LKOD
4,5,A,200077,Stavební povolení,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2005, 2006, 2007, 2008, 2009, 2010 ... 2020, 2...",OK_LKOD
5,6,A,270243,Výroba masa na jatkách,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021...",OK_LKOD
6,7,A,250180,Zaměstnaní a nezaměstnaní podle výsledků výběr...,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","1993, 1994, 1995, 1996, 1997, 1998 ... 2020, 2...",OK_LKOD
7,8,A,230057,Školy a školská zařízení,podpůrná / kontextová,True,vuzemi_cis,True,"2023, 2024","2006, 2007, 2008, 2009, 2010, 2011 ... 2020, 2...",OK_LKOD
8,9,A,270230,Hospodářská zvířata podle krajů,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2002, 2003, 2004, 2005, 2006, 2007 ... 2019, 2...",OK_LKOD
9,10,A,280041,Náklady na ochranu životního prostředí a ekono...,podpůrná / kontextová,True,uzemi_cis,True,"2023, 2024","2006, 2007, 2008, 2009, 2010, 2011 ... 2019, 2...",OK_LKOD


In [9]:

display_category_results(
    results,
    categories=DISPLAY_CATEGORIES,
    max_items=DISPLAY_LIMIT,
)


## 001. [A] Dokončené byty v obcích

,hodnota
polozka,
target_years_present,"2023, 2024"
has_primary_key,True
primary_key_column,vuzemi_cis


**HEAD**

,idhod,hodnota,stapro_kod,tb_cis,tb_kod,vuzemi_cis,vuzemi_kod,rok,casref_od,casref_do,stapro_txt,tb_txt,vuzemi_txt
0,1457433181,2,3103,NaN,NaN,43,539783,2024,2024-01-01,2024-12-31,Počet dokončených bytů,NaN,Oplot
1,1457432328,2,3103,NaN,NaN,43,532479,2024,2024-01-01,2024-12-31,Počet dokončených bytů,NaN,Kmetiněves
2,1457433488,1,3103,NaN,NaN,43,551821,2024,2024-01-01,2024-12-31,Počet dokončených bytů,NaN,Tvrdkov
3,1457431105,0,3103,NaN,NaN,43,559806,2024,2024-01-01,2024-12-31,Počet dokončených bytů,NaN,Hlohovice
4,1457433090,3,3103,NaN,NaN,43,574911,2024,2024-01-01,2024-12-31,Počet dokončených bytů,NaN,Dolní Roveň


**TAIL**

...

,hodnota
polozka,
target_years_present,None
has_primary_key,False
primary_key_column,None


**HEAD**

,VSTRANA,NAZEVCELK,NAZEV_STRV,ZKRATKAV30,ZKRATKAV8,POCSTR_SLO,SLOZENI,ZKRATKA_OF,TYPVS,PLNYNAZEV
0,1,KDU-ČSL,KDU-ČSL,KDU-ČSL,KDU-ČSL,1,001,KDU-ČSL,S,NaN
1,2,Česká strana národně sociální,Česká strana národně sociální,Česká strana národně sociální,ČSNS,1,002,ČSNS,S,NaN
2,3,Křesťanskodemokratická strana,Křesťanskodemokratická strana,Křesťanskodemokratická strana,KDS,1,003,KDS,S,NaN
3,4,Liberálně demokratická strana,Liberálně demokratická strana,Liberálně demokratická strana,LDS,1,004,LDS,S,NaN
4,5,Strana zelených,Strana zelených,Strana zelených,Zelení,1,005,Zelení,S,NaN


**TAIL**

,VSTRANA,NAZEVCELK,NAZEV_STRV,ZKRATKAV30,ZKRATKAV8,POCSTR_SLO,SLOZENI,ZKRATKA_OF,TYPVS,PLNYNAZEV
1428,1708,JSME,JSME,JSME,JSME,1,1708,JSME,S,NaN
1429,1709,Svěží Týniště,Svěží Týniště,Svěží Týniště,ST,1,1709,ST,S,NaN
1430,1710,Zbrojovka a fotbal zpět za Lužánky,Zbrojovka a fotbal zpět za Lužánky,Zbrojovka a fotbal za Lužánky,za Lužánky,1,1710,za Lužánky,S,NaN
1431,9995,Koalice,Koalice,Koalice,Koalice,1,9995,koalice,S,NaN
1432,9999,Neregistrováno u MV,Neregistrováno u MV,Neregistrováno u MV,Neregistr. u MV,1,9999,Neregistrováno u MV,S,NaN



## Co vznikne ve výstupní složce

Ve `czso_ikz_a_to_z_output` najdeš hlavně:

- `ikz_a_to_z_manifest.csv`
- `ikz_a_to_z_manifest.xlsx`
- `selected_relevant_catalogue.csv`
- `datasets_by_category.csv`
- `datasets_by_pk_and_years.csv`
- `datasets_by_status.csv`
- `ikz_a_to_z_previews.md`
- `previews/` – samostatné `head.csv`, `tail.csv`, `meta.json`
- `_downloads/` – dočasná složka pro stažené soubory; při výchozím nastavení se po vytvoření preview průběžně čistí

Poznámka k previewům:
- pokud je detekován řádkový časový sloupec (`rok`, `datum`, `casref`, `obdobi` apod.), jsou `head` a `tail` vytvořené **už z odfiltrovaných target years**
- pokud je čas jen v hlavičkách nebo není detekce spolehlivá, preview zůstane nefiltrované a důvod je zapsaný v `meta.json` / `notes`
